# String Functions

## Overview
String manipulation is essential for cleaning real-world data — provider names with inconsistent casing, ICD codes with extra characters, emails to parse, free-text fields to standardise.

**Core functions — SQLite and PostgreSQL:**

| Function | Purpose | SQLite | PostgreSQL |
|---|---|---|---|
| `LENGTH(s)` | Character count | ✓ | ✓ |
| `UPPER(s)` / `LOWER(s)` | Case conversion | ✓ | ✓ |
| `TRIM(s)` / `LTRIM` / `RTRIM` | Strip whitespace | ✓ | ✓ |
| `SUBSTR(s, start, len)` | Extract substring (1-indexed) | ✓ | `SUBSTRING(s, start, len)` |
| `INSTR(s, sub)` | Position of substring (0 if missing) | ✓ | `POSITION(sub IN s)` |
| `REPLACE(s, old, new)` | Substitute text | ✓ | ✓ |
| `s1 \|\| s2` | Concatenation | ✓ | ✓ (also `CONCAT()`) |
| `GROUP_CONCAT(s, sep)` | Aggregate to delimited string | ✓ | `STRING_AGG(s, sep ORDER BY ...)` |

**PostgreSQL extras:** `LEFT(s,n)`, `RIGHT(s,n)`, `LPAD`, `RPAD`, `SPLIT_PART`, `REGEXP_REPLACE`, `REGEXP_MATCHES`, `INITCAP`, `CONCAT_WS`

---

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cur  = conn.cursor()

cur.executescript("""
CREATE TABLE providers (
    provider_id   INTEGER PRIMARY KEY,
    full_name     TEXT,
    specialty     TEXT,
    npi_code      TEXT,      -- 10-digit identifier stored as text
    email         TEXT,
    province      TEXT
);

-- Deliberately messy names to practice cleaning
INSERT INTO providers VALUES
  (1,  ' Dr. Sarah  MacNeil ',  'Cardiology',         '1234567890', 's.macneil@health.ca',  'NB'),
  (2,  'dr. james wong',         'Oncology',           '0987654321', 'j.wong@health.ca',     'BC'),
  (3,  'DR. FATIMA AL-RASHID',   'Emergency Medicine', '1122334455', 'f.alrashid@health.ca', 'ON'),
  (4,  'Dr. Ethan Tremblay  ',   'Family Medicine',    '5544332211', 'e.tremblay@health.ca', 'QC'),
  (5,  'dr.noah williams',       'Cardiology',         '6677889900', 'n.williams@health.ca', 'AB'),
  (6,  'Dr. Mei Zhang',          'Radiology',          '9988776655', 'm.zhang@health.ca',    'ON'),
  (7,  ' DR PRIYA NAIR ',        'Oncology',           '4433221100', 'p.nair@health.ca',     'BC'),
  (8,  'Dr. Marcus Okafor',      'Family Medicine',    '1357924680', 'm.okafor@health.ca',   'NB');
""")
conn.commit()

print('providers table ready — names are intentionally messy:')
print(pd.read_sql('SELECT provider_id, full_name, specialty FROM providers', conn).to_string(index=False))


---
## TRIM, UPPER, LOWER, LENGTH


In [ ]:
# Basic cleaning: trim whitespace, standardise case
print('=== Name standardisation pipeline ===')
q = """
SELECT provider_id,
       full_name                              AS raw_name,
       TRIM(full_name)                        AS trimmed,
       UPPER(TRIM(full_name))                 AS uppercased,
       LOWER(TRIM(full_name))                 AS lowercased,
       LENGTH(TRIM(full_name))                AS name_length,
       LENGTH(full_name) - LENGTH(TRIM(full_name)) AS leading_trailing_spaces
FROM   providers
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Case-insensitive filter using LOWER
print()
print('=== Find cardiology providers (case-insensitive search) ===')
q2 = """
SELECT provider_id, TRIM(full_name) AS name, specialty, province
FROM   providers
WHERE  LOWER(specialty) = 'cardiology'
"""
print(pd.read_sql(q2, conn).to_string(index=False))


---
## SUBSTR, INSTR, and REPLACE


In [ ]:
# Extract username and domain from email
print('=== Parse email into username and domain ===')
q = """
SELECT email,
       INSTR(email, '@')                          AS at_position,
       SUBSTR(email, 1, INSTR(email, '@') - 1)   AS username,
       SUBSTR(email, INSTR(email, '@') + 1)       AS domain
FROM   providers
"""
print(pd.read_sql(q, conn).to_string(index=False))

# NPI code formatting: 1234567890 → 123-456-7890
print()
print('=== Format NPI codes as 123-456-7890 ===')
q2 = """
SELECT npi_code,
       SUBSTR(npi_code, 1, 3) || '-' ||
       SUBSTR(npi_code, 4, 3) || '-' ||
       SUBSTR(npi_code, 7, 4)         AS formatted_npi
FROM   providers
"""
print(pd.read_sql(q2, conn).to_string(index=False))

# REPLACE to clean up prefix variations
print()
print('=== Strip Dr./DR./dr. prefix ===')
q3 = """
SELECT full_name,
       TRIM(
         REPLACE(
           REPLACE(
             REPLACE(LOWER(TRIM(full_name)), 'dr. ', ''),
           'dr.', ''),
         'dr ', '')
       )  AS name_only
FROM   providers
"""
print(pd.read_sql(q3, conn).to_string(index=False))


---
## Concatenation and string building


In [ ]:
# Build formatted display strings
print('=== Formatted display strings ===')
q = """
SELECT provider_id,
       TRIM(full_name) || ' (' || specialty || ')'          AS display_name,
       UPPER(province) || ' — ' || TRIM(full_name)          AS province_roster_entry,
       'NPI: ' || SUBSTR(npi_code,1,3) || '-***-****'       AS masked_npi
FROM   providers
ORDER BY province, full_name
"""
print(pd.read_sql(q, conn).to_string(index=False))

# NULL in concatenation — the trap
print()
print('=== NULL propagation in || concatenation ===')
cur = conn.cursor()
cur.execute("INSERT INTO providers VALUES (9, NULL, 'Cardiology', '0000000001', NULL, 'NB')")
conn.commit()
q2 = """
SELECT provider_id,
       full_name || ' (' || specialty || ')'    AS concat_with_null,
       COALESCE(full_name,'[Unknown]') || ' (' || specialty || ')'  AS safe_concat
FROM   providers
WHERE  provider_id = 9
"""
print(pd.read_sql(q2, conn).to_string(index=False))

print()
print('=== PostgreSQL CONCAT() handles NULLs as empty strings automatically ===')
print("  CONCAT(full_name, ' (', specialty, ')')  -- NULL becomes '' not NULL")
print("  CONCAT_WS(' | ', col1, col2, col3)       -- WITH SEPARATOR, skips NULLs")


---
## GROUP_CONCAT / STRING_AGG — aggregating strings


In [ ]:
# GROUP_CONCAT — aggregate providers into comma-separated list per specialty
print('=== Providers grouped by specialty (SQLite GROUP_CONCAT) ===')
q = """
SELECT specialty,
       COUNT(*)                                                  AS n_providers,
       GROUP_CONCAT(TRIM(full_name), ' | ')                     AS provider_list,
       GROUP_CONCAT(DISTINCT province)                          AS provinces
FROM   providers
WHERE  provider_id <> 9   -- exclude the NULL-name test row
GROUP BY specialty
ORDER BY n_providers DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))

print()
print('=== PostgreSQL STRING_AGG (supports ORDER BY inside aggregate) ===')
print("""
  SELECT specialty,
         STRING_AGG(TRIM(full_name), ' | ' ORDER BY full_name) AS provider_list
  FROM   providers
  GROUP BY specialty;

  -- Unlike GROUP_CONCAT, STRING_AGG supports ORDER BY inside the aggregate.
  -- GROUP_CONCAT order in SQLite is implementation-defined — pre-sort with a subquery
  -- if consistent ordering matters.
""")

conn.close()


---
## Common Pitfalls

**1. `||` concatenation returns NULL if any operand is NULL**
`first_name || ' ' || last_name` returns NULL when either name is NULL. Wrap with `COALESCE`: `COALESCE(first_name,'') || ' ' || COALESCE(last_name,'')`. PostgreSQL's `CONCAT()` treats NULLs as empty strings automatically — prefer it for name assembly.

**2. LIKE case sensitivity differs between SQLite and PostgreSQL**
In SQLite, `LIKE` is case-insensitive for ASCII characters by default — `LIKE 'cardiology'` matches `'Cardiology'`. In PostgreSQL, `LIKE` is case-sensitive — use `ILIKE` for case-insensitive matching or `LOWER(col) LIKE LOWER(pattern)` for portability.

**3. SUBSTR uses 1-based indexing**
`SUBSTR('2023-01-15', 1, 4)` returns `'2023'`. Position 1 is the first character. Programmers used to 0-indexed languages often write `SUBSTR(s, 0, 4)` — in SQLite this still works (position 0 is treated as 1), but it's non-standard and confusing. Always use 1-based positions.

**4. INSTR returns 0, not NULL, when the substring is not found**
`INSTR('hello', 'z')` returns 0. If you use `INSTR(email, '@') - 1` in a SUBSTR call and `@` is missing, you get `SUBSTR(email, 0-1, ...)` which returns an empty string rather than an error. Guard with `CASE WHEN INSTR(email, '@') > 0 THEN ... END`.

**5. GROUP_CONCAT order is non-deterministic in SQLite**
`GROUP_CONCAT(name, ',')` returns names in an undefined order. If consistent ordering matters, wrap in a subquery that sorts first, or use PostgreSQL's `STRING_AGG(name, ',' ORDER BY name)`. Don't rely on incidental ordering from your current data.

**6. TRIM only removes spaces by default**
`TRIM(full_name)` removes leading and trailing spaces but not tabs (`\t`), newlines (`\n`), or other whitespace. If your source data contains these, use `REPLACE` chains or PostgreSQL's `REGEXP_REPLACE` to clean all whitespace variants.


---
*sql_methods_library - Samantha McGarrigle*